# ForgeLM — Multi-Dataset Training

Mix multiple datasets with configurable ratios in a single training run.

**Requirements:** GPU runtime recommended (Runtime → Change runtime type → T4 GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemililik/ForgeLM/blob/main/notebooks/multi_dataset.ipynb)

In [ ]:
# Step 1: Install ForgeLM (with bitsandbytes for 4-bit quantization)
# Pinned to v0.5.7; bump on each release
!pip install -q --no-cache-dir 'forgelm[qlora]==0.5.7' bitsandbytes
!pip uninstall -y wandb -q
# Colab base image ships torchao 0.10; modern peft requires >=0.16 for PeftModel.from_pretrained.
# Upgrade so the comparison cell at the end can load the trained LoRA adapter.
!pip install -q --upgrade 'torchao>=0.16.0'
!forgelm --version

In [ ]:
# Step 2: Check GPU
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 1)
    print(f"GPU: {gpu_name} ({vram_gb} GB VRAM)")
else:
    print("WARNING: No GPU. Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# Step 3: Create two datasets
import json

# Dataset 1: General Q&A
general = [
    {
        "messages": [
            {"role": "user", "content": "What is AI?"},
            {
                "role": "assistant",
                "content": "AI is artificial intelligence — the simulation of human intelligence by machines.",
            },
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is ML?"},
            {"role": "assistant", "content": "ML is machine learning — a subset of AI where systems learn from data."},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is deep learning?"},
            {
                "role": "assistant",
                "content": "Deep learning uses neural networks with many layers to learn complex patterns from large amounts of data.",
            },
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is NLP?"},
            {
                "role": "assistant",
                "content": "NLP is natural language processing — AI that understands and generates human language.",
            },
        ]
    },
]

# Dataset 2: Domain-specific (coding)
coding = [
    {
        "messages": [
            {"role": "user", "content": "Write hello world in Python"},
            {"role": "assistant", "content": "print('Hello, World!')"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is a list?"},
            {"role": "assistant", "content": "A list is a mutable ordered sequence in Python: my_list = [1, 2, 3]"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "How to read a file?"},
            {"role": "assistant", "content": "Use: with open('file.txt') as f: content = f.read()"},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is a dictionary?"},
            {
                "role": "assistant",
                "content": "A dictionary is a key-value mapping in Python: my_dict = {'name': 'Alice', 'age': 30}",
            },
        ]
    },
]

for name, data in [("general.jsonl", general), ("coding.jsonl", coding)]:
    with open(name, "w") as f:
        for item in data:
            f.write(json.dumps(item) + "\n")

print(f"Created: general.jsonl ({len(general)} samples), coding.jsonl ({len(coding)} samples)")

In [ ]:
# Step 4: Create config with multi-dataset mixing
config_yaml = """
model:
  name_or_path: "HuggingFaceTB/SmolLM2-135M-Instruct"
  max_length: 1024
  load_in_4bit: true

lora:
  r: 16
  alpha: 32
  target_modules: ["q_proj", "v_proj"]

training:
  output_dir: "./multi_checkpoints"
  max_steps: 50
  per_device_train_batch_size: 4
  learning_rate: 2.0e-5

data:
  dataset_name_or_path: "general.jsonl"
  extra_datasets:
    - "coding.jsonl"
  mix_ratio: [0.7, 0.3]  # 70% general, 30% coding
"""

with open("multi_config.yaml", "w") as f:
    f.write(config_yaml)
print("Multi-dataset config saved! (70% general, 30% coding, max_steps: 50)")

In [ ]:
# Step 5: Validate
!forgelm --config multi_config.yaml --dry-run

In [ ]:
# Step 6: Train
!forgelm --config multi_config.yaml

In [ ]:
# Step 7: Verify output
import os

model_path = "./multi_checkpoints/final_model"
if not os.path.exists(model_path):
    print(f"ERROR: '{model_path}' not found. Check training output above.")
elif not os.path.isfile(os.path.join(model_path, "adapter_config.json")):
    print(f"ERROR: adapter_config.json not found in '{model_path}'.")
else:
    print(f"Training completed! Model saved to: {model_path}")
    print(f"Files: {os.listdir(model_path)}")

In [ ]:
# Step 8: Compare base model vs multi-dataset fine-tuned model
import os

from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

model_path = "./multi_checkpoints/final_model"
base_model_name = "HuggingFaceTB/SmolLM2-135M-Instruct"

if not os.path.exists(model_path) or not os.path.isfile(os.path.join(model_path, "adapter_config.json")):
    print("Error: Model not found. Ensure training completed successfully.")
else:
    print("Loading base model...")
    base_model = AutoModelForCausalLM.from_pretrained(base_model_name)
    tokenizer = AutoTokenizer.from_pretrained(base_model_name)

    print("Loading multi-dataset fine-tuned model...")
    ft_model = PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained(base_model_name), model_path)
    ft_tokenizer = AutoTokenizer.from_pretrained(model_path)

    # Test both domains + unseen prompt
    test_prompts = [
        "What is reinforcement learning?",  # general domain (unseen)
        "How do I sort a list in Python?",  # coding domain (unseen)
        "Explain what a neural network is.",  # cross-domain (unseen)
    ]

    for prompt in test_prompts:
        messages = [{"role": "user", "content": prompt}]
        formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(formatted, return_tensors="pt")

        print(f"\n{'=' * 60}")
        print(f"PROMPT: {prompt}")
        print(f"{'=' * 60}")

        base_out = base_model.generate(**inputs, max_new_tokens=200, do_sample=True, temperature=0.7)
        base_resp = tokenizer.decode(base_out[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True).strip()
        print(f"\n[BASE MODEL]:\n{base_resp[:400]}")

        ft_inputs = ft_tokenizer(formatted, return_tensors="pt")
        ft_out = ft_model.generate(**ft_inputs, max_new_tokens=200, do_sample=True, temperature=0.7)
        ft_resp = ft_tokenizer.decode(ft_out[0][ft_inputs["input_ids"].shape[1] :], skip_special_tokens=True).strip()
        print(f"\n[FINE-TUNED]:\n{ft_resp[:400]}")

    print(f"\n{'=' * 60}")
    print("Fine-tuned model learned from BOTH datasets (70% general + 30% coding).")

## Step 9: Download or push your trained adapter

The training cell above wrote a PEFT adapter (`adapter_config.json` +
`adapter_model.safetensors` + tokenizer files, ~30-100 MB total). Two
ways to get it off Colab:

- **Local download** — ZIP the adapter and trigger a browser download.
  The ZIP lands on your filesystem; extract anywhere and load back via
  `PeftModel.from_pretrained(base, "./extracted_folder")`.
- **HuggingFace Hub push** — push directly to a repo for sharing /
  versioning. Requires `HF_TOKEN` in Colab Secrets with write access to
  the target repo. Uncomment the **Option B** block.

If your Colab session is about to time out, run this cell first — the
adapter doesn't survive a runtime disconnect.

In [ ]:
# Option A — ZIP + Colab browser download (default)
import os
import shutil
from pathlib import Path

MODEL_PATH = "./multi_checkpoints/final_model"
ZIP_NAME = "smollm2_multi_finetune"

if not Path(MODEL_PATH).is_dir():
    print(f"No trained adapter at {MODEL_PATH}. Run the training cell above first.")
else:
    archive = shutil.make_archive(ZIP_NAME, "zip", root_dir=MODEL_PATH)
    archive_size = Path(archive).stat().st_size
    print(f"Created {archive} ({archive_size:,} bytes)")
    try:
        from google.colab import files  # type: ignore

        files.download(archive)
        print("Browser download triggered.")
    except ImportError:
        print(f"Local Jupyter — grab the ZIP from {Path(archive).resolve()}")

# ---------------------------------------------------------------------------
# Option B — Push to HuggingFace Hub (uncomment + edit REPO_ID to use)
# ---------------------------------------------------------------------------
# Set HF_TOKEN in Colab → 🔑 Secrets first (with write access to your repo).
#
# from huggingface_hub import HfApi
# try:
#     from google.colab import userdata
#     os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
# except Exception:
#     pass
#
# REPO_ID = "your-username/smollm2_multi_finetune"  # change to your HF Hub repo
# api = HfApi()
# api.create_repo(repo_id=REPO_ID, exist_ok=True, private=False)
# api.upload_folder(folder_path=MODEL_PATH, repo_id=REPO_ID, repo_type="model")
# print(f"Pushed to https://huggingface.co/{REPO_ID}")